# Delta Live Tables - Streaming vs Batch

---

## 📊 Streaming Table vs Materialized View

A escolha entre Streaming Table e Materialized View depende do **tipo de processamento** e **fonte de dados**.

```
┌─────────────────────────────────────────────────────────────────┐
│                      STREAMING TABLE                            │
│  ───────────────────────────────────────────────────────────    │
│  • Processa cada linha UMA ÚNICA VEZ                            │
│  • Fonte: append-only (novos dados apenas)                      │
│  • Não recomputa dados antigos                                  │
│  • Ideal para: Ingestão, transformações incrementais            │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│                     MATERIALIZED VIEW                           │
│  ───────────────────────────────────────────────────────────    │
│  • Pode RECOMPUTAR dados quando necessário                      │
│  • Fonte: pode ler múltiplas vezes                              │
│  • Suporta agregações e queries complexas                       │
│  • Ideal para: Gold layer, relatórios, dashboards               │
└─────────────────────────────────────────────────────────────────┘
```

---

## 🔄 Comparação Detalhada

| Característica | Streaming Table | Materialized View |
|----------------|-----------------|-------------------|
| **Processamento** | Incremental (append) | Batch (pode recomputar) |
| **Cada linha processada** | Uma única vez | Pode ser múltiplas vezes |
| **Mudança na query** | Não reprocessa dados antigos | Recomputa tudo |
| **Fonte de dados** | Append-only | Qualquer |
| **Agregações** | Limitadas | Completas |
| **Updates/Deletes na fonte** | ❌ Não suporta | ✅ Suporta |
| **Layer típico** | Bronze, Silver | Gold |

---

## 📥 Streaming Table - Sintaxe

### SQL

```sql
-- Sintaxe básica
CREATE OR REFRESH STREAMING TABLE table_name
AS SELECT * FROM STREAM(source)

-- Com Auto Loader (ingestão de arquivos)
CREATE OR REFRESH STREAMING TABLE bronze_orders
COMMENT "Raw orders from cloud storage"
AS SELECT 
    *,
    current_timestamp() as ingestion_time,
    input_file_name() as source_file
FROM cloud_files("/mnt/raw/orders/", "json")

-- Lendo de outra Streaming Table do pipeline
CREATE OR REFRESH STREAMING TABLE silver_orders
AS SELECT 
    CAST(order_id AS INT) as order_id,
    CAST(amount AS DECIMAL(10,2)) as amount
FROM STREAM(LIVE.bronze_orders)
```

### Python

```python
import dlt

# Sintaxe básica
@dlt.table(name="table_name")
def my_table():
    return dlt.read_stream("source")

# Com Auto Loader
@dlt.table(
    name="bronze_orders",
    comment="Raw orders from cloud storage"
)
def bronze_orders():
    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "json")
            .load("/mnt/raw/orders/")
            .withColumn("ingestion_time", current_timestamp())
    )

# Lendo de outra Streaming Table
@dlt.table(name="silver_orders")
def silver_orders():
    return (
        dlt.read_stream("bronze_orders")
            .selectExpr(
                "CAST(order_id AS INT) as order_id",
                "CAST(amount AS DECIMAL(10,2)) as amount"
            )
    )
```

---

## 📊 Materialized View - Sintaxe

### SQL

```sql
-- Sintaxe básica (batch read)
CREATE OR REFRESH MATERIALIZED VIEW view_name
AS SELECT * FROM source

-- Com agregação (típico Gold layer)
CREATE OR REFRESH MATERIALIZED VIEW gold_daily_sales
COMMENT "Daily sales summary"
AS SELECT
    order_date,
    COUNT(*) as total_orders,
    SUM(amount) as total_revenue,
    AVG(amount) as avg_order_value
FROM LIVE.silver_orders
GROUP BY order_date

-- JOIN entre tabelas
CREATE OR REFRESH MATERIALIZED VIEW gold_customer_orders
AS SELECT 
    c.customer_name,
    COUNT(o.order_id) as num_orders,
    SUM(o.amount) as total_spent
FROM LIVE.silver_orders o
JOIN LIVE.silver_customers c ON o.customer_id = c.customer_id
GROUP BY c.customer_name
```

### Python

```python
import dlt
from pyspark.sql.functions import sum, count, avg

# Sintaxe básica
@dlt.table(name="view_name")
def my_view():
    return dlt.read("source")  # Nota: read() não read_stream()

# Com agregação
@dlt.table(
    name="gold_daily_sales",
    comment="Daily sales summary"
)
def gold_daily_sales():
    return (
        dlt.read("silver_orders")
            .groupBy("order_date")
            .agg(
                count("*").alias("total_orders"),
                sum("amount").alias("total_revenue"),
                avg("amount").alias("avg_order_value")
            )
    )

# JOIN entre tabelas
@dlt.table(name="gold_customer_orders")
def gold_customer_orders():
    orders = dlt.read("silver_orders")
    customers = dlt.read("silver_customers")
    
    return (
        orders.join(customers, "customer_id")
            .groupBy("customer_name")
            .agg(
                count("order_id").alias("num_orders"),
                sum("amount").alias("total_spent")
            )
    )
```

---

## 🔑 Funções de Referência

| Função | Tipo | SQL | Python |
|--------|------|-----|--------|
| **Streaming read** | Incremental | `STREAM(LIVE.table)` | `dlt.read_stream("table")` |
| **Batch read** | Completo | `LIVE.table` | `dlt.read("table")` |
| **Auto Loader** | Streaming | `cloud_files(path, format)` | `spark.readStream.format("cloudFiles")` |

### Regra de Ouro

```
┌─────────────────────────────────────────────────────────────────┐
│  STREAMING TABLE  ──────────►  Use STREAM() ou read_stream()    │
│  MATERIALIZED VIEW ─────────►  Use LIVE. ou read()              │
└─────────────────────────────────────────────────────────────────┘
```

---

## 👁️ Views Temporárias

Views **não armazenam dados** - úteis para queries intermediárias ou validação.

### SQL

```sql
CREATE TEMPORARY LIVE VIEW valid_orders
AS SELECT * FROM LIVE.bronze_orders
WHERE order_id IS NOT NULL AND amount > 0
```

### Python

```python
@dlt.view(name="valid_orders")
def valid_orders():
    return (
        dlt.read("bronze_orders")
            .filter("order_id IS NOT NULL AND amount > 0")
    )
```

> 💡 Use `TEMPORARY` para tabelas que não precisam ser publicadas no schema.

---

## 🔗 Referências

- [Transformar dados com DLT](https://learn.microsoft.com/pt-br/azure/databricks/delta-live-tables/transform)
- [Streaming Tables](https://learn.microsoft.com/en-us/azure/databricks/delta-live-tables/streaming)
- [Materialized Views](https://learn.microsoft.com/en-us/azure/databricks/sql/user/materialized-views)